<h1 style="text-align: center; vertical-align: middle;">S35 Accelerator & Detector Physics</h1>
<h2 style="text-align: center; vertical-align: middle;">Accelerator Physics by Professor Adrian Oeftiger</h2>

<div style="width: 45%; margin: auto; margin-top: 1em; padding-bottom: 1em; vertical-align: middle;">
<img src="./img/University_of_Oxford_logo.png" style="width: 40%; float: left; margin-right: 5%;" /><img src="./img/jai-logo.png" style="width: 50%; float: left; margin-right: 5%; margin-top:0;" />
</div>

<h3 style="clear: both; text-align: center; margin-top: 3em;">Lecture 6: Off-momentum Effects, Synchrotron Radiation</h3>

<h2>Run this notebook online!</h2>

Interact and run this jupyter notebook online:

<div class="alert alert-block alert-info" style="text-align:center;">
via the local Physics Jupyterlab service: <br />
<a href="https://jupyterlab.physics.ox.ac.uk" style="width:auto; display:table;margin:0.5em auto;"><img src="https://shields.io/badge/JupyterLab-Department%20of%20Physics-F37626?logo=jupyter" alt="physics department logo" style="height: 1.3em;" /></a>
</div>

<div class="alert alert-block alert-info" style="text-align:center;">
via the public mybinder.org service: <br />
<a href="https://mybinder.org/v2/gh/aoeftiger/UOXF-AP-2026/v6.1" style="width:auto; display:table;margin:0.5em auto;"><img src="./img/binder_logo.svg" alt="mybinder.org logo" style="height: 1.3em;" /></a>
</div>

Also find this lecture rendered [as HTML slides on github $\nearrow$](https://aoeftiger.github.io/UOXF-AP-2026/) along with the [source repository $\nearrow$](https://github.com/aoeftiger/UOXF-AP-2026).

<h2>Run this first!</h2>

Imports and modules:

In [ ]:
from config import (np, plt, sys, Madx, interp1d, PyNAFF,
                    pysixtrack, elements,
                    M_drift, M_dip_x, M_dip_y, 
                    M_quad_x, M_quad_y, 
                    track, track_sext_4D)
%matplotlib inline

<h2 style="color: #b51f2a">Refresher!</h2>

1. Hill Differential Equation & Floquet Theory
2. Twiss Parametrisation and Optics
3. The FODO Cell

<h2 style="color: #b51f2a">Today!</h2>

1. Off-momentum Particles: Dispersion & Chromaticity
2. Synchrotron Radiation


<!--
- synchrotron radiation power loss
- FCCee
- light sources
- FEL, XFEL
- BSRT in LHC detectors
-->

<!--research happening at JAI: advanced acceleration technologies (plasma accelerators), intense and high-energy beam physics, medical accelerators, beam diagnostics-->

<div style="text-align: center; width:100%;">
    <h2>Part I: Off-momentum Particles, Dispersion & Chromaticity</h2>
</div>

<h2>Off-momentum Trajectory</h2>

<img src="./img/dispersive_orbit_in_dipole.png" alt="Dispersive orbit in dipole magnet" style="float:right; width:20%; margin-left: 1em;"/>

Particles with a momentum $p\neq p_0$ different from reference momentum $p_0$ are bent by magnetic fields  $B$ on a different bending radius $\rho$:

$$B\rho=\cfrac{p}{|q|}$$

In particular, higher-momentum particles are bent less, and their off-momentum trajectory encompasses two major effects:

1. Dispersion (dipole magnets, change of closed orbit)
2. Chromaticity (quadrupole magnets,  change of focusing)

<p style="clear: both; font-size: 10pt; text-align: right; float: right;">
image by <a href="https://people.nscl.msu.edu/~haoy/teaching/fundamental_AP/transverse_dynamics/transverse_dynamics.html">Y. Hao</a></p>

<h2>Dispersion</h2>

<img src="./img/dispersion.png" alt="Sketch showing dispersion effect on closed orbit" style="width:25%; float: right; margin: 1em;" />

Since dipole magnets define the reference orbit (cf. Frenet-Serret coordinate system), off-momentum particles will experience a different "dispersive" closed orbit &ndash; dipoles are said to "generate" dispersion.

The dispersion function $D(s)$ describes the local equilibrium offset from the reference closed orbit due to a momentum deviation $\delta=\Delta p/p_0$ with respect to the synchronous reference particle:

$$x(s)=\underbrace{x_\beta}\limits_{\mathop{=}\sqrt{2J_x\beta_x(s)}\cos(\psi_x(s))} + x_\text{disp}\qquad\text{with}\qquad x_\text{disp}(s) = D_x(s) \cdot \delta$$

$D_x(s)$ hence represents the (horizontal) closed orbit for a particle with $\delta=1$, i.e. twice the reference momentum.

<p style="clear: both; font-size: 10pt; text-align: right; float: right;">image by <a href="https://e-publishing.cern.ch/index.php/CYRSP/article/view/1586/1315">A. Lasheen / JUAS</a></p>

### Simulation 1: Computing the Dispersion Function of a FODO cell

For illustration, we use again the LHC FODO cell:

In [ ]:
madx = Madx(stdout=sys.stdout)

madx.input('''
k1l_f := 0.008 * 3.3;  // inverse focal length qf
k1l_d := -0.008 * 3.3; // inverse focal length qd
theta := 0;            // in LHC: 2 * pi / 1232;

qf2: quadrupole, l = 3.3 / 2, k1 := k1l_f / 3.3; // half a focusing quad
qd:  quadrupole, l = 3.3,     k1 := k1l_d / 3.3;
dip: sbend, l = 14.3, angle := theta;

fodo: sequence, l = 110;
qf2, at = 3.3 / 4;
dip, at = 12; 
dip, at = 2 * 110 / 8; 
dip, at = 110 / 2 - 12; 
qd, at = 110 / 2;
dip, at = 110 / 2 + 12; 
dip, at = 6 * 110 / 8; 
dip, at = 110 - 12; 
qf2, at = 110 - 3.3 / 4;
endsequence;
''');

madx.command.beam(particle='proton', energy=7e3) # energy is in GeV!
madx.use(sequence='fodo')

# output the Twiss parameters every 1m
madx.command.select(flag="interpolate", sequence="fodo", step=1);

twiss = madx.twiss();
qx_fodo = twiss.summary['q1']
qpx_fodo = twiss.summary['dq1'] # multiplied by beta~1

In [ ]:
madx.input('value, beam->beta;')

We "switch" on the dipole magnets by defining a non-zero bending angle:

In [ ]:
madx.input('theta := 2 * pi / 1232;');

Let us recompute the optics functions, as this time also the dispersion function will assume finite values (<span style="color: #e6541a;">$\leadsto$ why?</span>):

In [ ]:
twiss = madx.twiss();

Given that $D=\cfrac{x}{\delta}$ for the equilibrium orbit at a momentum deviation $\delta$, the dispersion function $D(s)$ satisfies the inhomogeneous Hill differential equation

$$D'' + \left(\frac{1}{\rho_0(s)} + k(s)\right)\cdot D = \frac{1}{\rho_0(s)}$$

In [ ]:
plt.plot(twiss['s'], twiss['dx'] * 0.999999991) # multiplied by beta~1
plt.xlabel('$s$ [m]')
plt.ylabel('$D_x(s)$ [m]');

$\implies$ Dispersion function $D(s)$ is focused by the quadrupoles, analogous to the horizontal $\beta_x(s)$-function!

<p style="color: #e6541a;">$\leadsto$ Verify that the dispersion is generated by the dipole magnets! (What does this mean?) How does the dispersion function $D(s)$ look like when the dipole fields are switched off?</p>

### Simulation 2: Dispersion Effect in Tracking

To illustrate the dispersion effect, we use the thin-lens tracking code `PySixTrack` (like our thin-lens betatron matrices but for 6D, i.e. including the momentum deviation $\delta$)!

We define a drift of $5\,$m length and a dipole with a bending angle of $0.1\,$rad:

In [ ]:
drift = elements.DriftExact(5)
dipole = elements.Multipole(knl=[0.1], hxl=0.1)

Initialise two particles, both at $x=0.04\,$m but only one at a momentum deviation of $\delta=10^{-3}$:

In [ ]:
part0 = pysixtrack.Particles(x=0, delta=0)
part1 = pysixtrack.Particles(x=0, delta=0.001)

Track through the drift, then the dipole and again the drift:

In [ ]:
rec_x0 = [part0.x]
rec_x1 = [part1.x]

drift.track(part0)
drift.track(part1)

rec_x0 += [part0.x]
rec_x1 += [part1.x]

dipole.track(part0)
dipole.track(part1)

rec_x0 += [part0.x]
rec_x1 += [part1.x]

drift.track(part0)
drift.track(part1)

rec_x0 += [part0.x]
rec_x1 += [part1.x]

In [ ]:
plt.plot([0, 5, 5, 10], rec_x0, label='$\delta=0$')
plt.plot([0, 5, 5, 10], rec_x1, label='$\delta=10^{-3}$')

plt.xlabel('$s$ [m]')
plt.ylabel('$x$ [m]')
plt.legend();

<p style="color: #e6541a;">$\leadsto$ Please explain what happens here and why.</p>

<h2>Chromaticity</h2>

Chromaticity describes the change of focusing for off-momentum particles at $p=p_0(1+\delta)$. <br />
The chromatic aberration of a quadrupole focusing lens with gradient $\frac{\partial B_y}{\partial x}$ yields:

$$k \equiv \cfrac{1}{B\rho}\cfrac{\partial B_y}{\partial x} \propto \cfrac{1}{p}\implies k(\delta)=\cfrac{k}{1+\delta}$$

Therefore, quadrupoles focus less for higher-momentum particles $\delta>0$ and thus the tune $Q$ decreases. The chromaticity of an accelerator is defined as

$$Q'=\frac{dQ}{d\delta}$$

The natural chromaticity of a (linear) magnetic lattice is typically negative.

### Simulation 3: Chromaticity Effect in Tracking

Let us illustrate by tracking particles in `PySixTrack` again. We define a quadrupole with an integrated focusing strength of $k\cdot L=0.3\,$m${}^{-1}$:

In [ ]:
quad = elements.Multipole(knl=[0, 0.3])

<p style="color: #e6541a;">$\leadsto$ What is the expected focal length $f=\cfrac{1}{k\cdot L}$?</p>

Initialise two sets of particles with the same distribution in $x$, one of which features a momentum deviation of $\delta=0.1$ (!):

<p style="color: #e6541a;">$\leadsto$ What is the expected focal length change $\Delta f(\delta)$ for these $\delta=0.1$ particles?</p>

In [ ]:
npart = 11
x_dist = np.linspace(-0.05, 0.05, npart)

In [ ]:
part0 = pysixtrack.Particles(x=x_dist.copy(), delta=0)
part1 = pysixtrack.Particles(x=x_dist.copy(), delta=0.1)

Track through the $5\,$m drift, then through the quadrupole and again through the same drift:

In [ ]:
rec_x0 = [part0.x.copy()]
rec_x1 = [part1.x.copy()]

drift.track(part0)
drift.track(part1)

rec_x0 += [part0.x.copy()]
rec_x1 += [part1.x.copy()]

quad.track(part0)
quad.track(part1)

rec_x0 += [part0.x.copy()]
rec_x1 += [part1.x.copy()]

drift.track(part0)
drift.track(part1)

rec_x0 += [part0.x.copy()]
rec_x1 += [part1.x.copy()]

In [ ]:
for i in range(npart):
    l0, = plt.plot([0, 5, 5, 10], np.array(rec_x0)[:, i], c='C0', lw=1)
    l1, = plt.plot([0, 5, 5, 10], np.array(rec_x1)[:, i], c='C1', lw=1)

plt.xlabel('$s$ [m]')
plt.ylabel('$x$ [m]')
plt.legend([l0, l1], ['$\delta=0$', '$\delta=0.1$']);

$\implies$ We observe less focusing for $\delta>0$ particles $\implies$ less phase advance in quasi-harmonic oscillation $\implies$ negative tune shift!

<h2>Natural Chromaticity of a FODO Cell</h2>

Natural chromaticity of a FODO cell with phase advance $\Phi_\text{FODO}$ can be derived via *thin-lens approximation* (<a href="https://aoeftiger.github.io/UOXF-AP-2026/lecture4/lecture.slides.html#/27">lecture 4</a>):
    
$$Q'_\text{FODO} = -\frac{1}{\pi}\tan\left(\frac{\Phi_\text{FODO}}{2}\right)$$

Natural chromaticity is always negative (<span style="color: #e6541a;">$\leadsto$ why?</span>).

### Simulation 4: Chromatic Detuning in a FODO Cell from Tracking

Let us track with `PySixTrack` through the LHC FODO cell for a distribution of particles with momentum spread and observe the chromatic tune shift.

We first define the same FODO cell as in `MAD-X` before (just in thin-lens approximation and without dipoles):

In [ ]:
kL = 0.008 * 3.3

qf2_fodo = elements.Multipole(knl=[0, kL / 2.])
qd_fodo = elements.Multipole(knl=[0, -kL])
drift_fodo = elements.DriftExact(110 / 2.)

fodo = [qf2_fodo, drift_fodo, qd_fodo, drift_fodo, qf2_fodo]

We initialise a distribution of `npart` macro-particles, with a momentum spread between $\delta\in[-10^{-3},10^{-3}]$ at a fixed initial horizontal position of $x=0.04\,$m:

In [ ]:
npart = 21
x_ini = 0.04
delta = np.linspace(-0.001, 0.001, npart)

particles = pysixtrack.Particles(x=x_ini, delta=delta)

We will record the $x$ position after each FODO cell for each particle:

In [ ]:
ncells = 1024

rec_x = np.zeros((ncells, npart), dtype=float)
rec_x[0] = particles.x

Let's go for the tracking:

In [ ]:
for i in range(1, ncells):
    for el in fodo:
        el.track(particles)
    rec_x[i] = particles.x

Comparing two particles with same initial $x$ but different $\delta$, we already see the different phase advance in the recorded horizontal motion:

In [ ]:
plt.plot(rec_x[:, npart//2 + 1], lw=2, label='$\delta=0$')
plt.plot(rec_x[:, 0], lw=2, label='$\delta=-0.001$')

plt.xlabel('number of FODO cells')
plt.ylabel('$x$ [m]')
plt.legend(bbox_to_anchor=(1.05, 1));

Let's evaluate the tune of each particle using the NAFF algorithm:

In [ ]:
Qx_delta = np.zeros(npart, dtype=float)

for i in range(npart):
    Qx_delta[i] = PyNAFF.naff(rec_x[:, i], turns=ncells, nterms=1)[0, 1]

The tune of the $\delta=0$ particle should be the tune of the reference particle in this linear lattice:

In [ ]:
Qx_delta[npart//2 + 1]

In [ ]:
qx_fodo

In [ ]:
plt.plot(1e3 * delta, Qx_delta)

plt.xlabel('$\delta$ [$10^{-3}$]')
plt.ylabel('$Q_x$');

$\implies$ The tune changes with the momentum, as anticipated! The slope of this line is the first-order chromaticity $Q'_x$!

The `numpy` function `polyfit` is useful for a quick linear regression, the output is $(a,b)$ for $y=a\cdot x + b$:

In [ ]:
np.polyfit(delta, Qx_delta, 1)

The slope and thus the chromaticity of the LHC FODO cell is approximately $Q'_x=-0.3$, measured via particle tracking. 

#### Comparison

The analytic formula 
$Q'_\text{FODO} = -\frac{1}{\pi}\tan\left(\frac{\Phi_\text{FODO}}{2}\right)$
gives:

In [ ]:
-1 / np.pi * np.tan(2 * np.pi * qx_fodo / 2)

`MAD-X` would have given us this value, too, we evaluated it as `twiss.summary['dq1'] * beta` (where `beta` is the speed of the particles):

In [ ]:
qpx_fodo

<h2>Bonus: Chromaticity Correction with Sextupoles</h2>

On average over many turns (during which $\delta$ remains more or less constant &ndash; remember synchrotron motion is much slower than tranverse betatron motion), the horizontal position of the particles corresponds to their dispersive closed orbit:
    
$$\langle x \rangle = \langle x_\beta + x_\text{disp}\rangle = \langle \sqrt{2J\beta_x}\underbrace{\cos(\psi_x+\psi_{x0}) }\limits_{\langle\cos\rangle\mathop{=}0}\rangle + \langle D\cdot \delta\rangle = D\cdot \delta$$

Sextupole magnets provide a means to correct for chromaticity in a location where $D(s)\neq 0$ such that particles are on average sorted by momentum $\delta$.

### Bonus Simulation 5: Chromaticity Correction in a FODO Cell

For demonstration, we add sextupoles to the FODO lattice in `MAD-X` and compute their necessary strength.

Make sure that dipoles are switched on, the dipole angle `theta` should be non-zero:

In [ ]:
madx.input('value, theta;')

(Otherwise re-run the notebook in order, you need the line `madx.input('theta := 2 * pi / 1232;')`)

We add two sextupole magnets, one next to each quadrupole:

In [ ]:
madx.input('sext1: sextupole, l = 1, k2 := k2sext1;')
madx.input('sext2: sextupole, l = 1, k2 := k2sext2;')
madx.command.seqedit(sequence='fodo')
madx.command.install(element='sext1', at=3.3/2 + 1)
madx.command.install(element='sext2', at=110/2 + 3.3/2 + 1)
madx.command.endedit()

In [ ]:
madx.use('fodo')

In [ ]:
madx.input(
'''match, sequence = fodo;
global, sequence = fodo, dq1 = {Qpx}, dq2 = {Qpy};
vary, name = k2sext1, step = 0.0001;
vary, name = k2sext2, step = 0.0001;
lmdif, tolerance = 1e-12;
endmatch;
'''.format(Qpx=0, Qpy=0))

$\implies$ The iterative algorithm in `MAD-X` has found suitable values of the sextupole strengths `k2sext1` and `k2sext2` such that the target chromaticity has been corrected to 0.

In [ ]:
twiss = madx.twiss();

twiss.summary.dq1 # this is the chromaticity

$\implies$ The chromaticity from the `MAD-X` optics computation has really become (numerically) zero!

We return to the `PySixTrack` tracking: after adding sextupoles of these strengths and the dipole magnets, we should be able to see the change via evaluating the chromaticity via NAFF from tracking data again!

As a short cut, we make the `MAD-X` lattice "thin" (apply thin-lens approximation) and transfer the lattice with dipoles and sextupoles to `PySixTrack`:

In [ ]:
assert madx.command.select(
    flag='MAKETHIN',
    class_='quadrupole',
    slice_=1,
)

assert madx.command.select(
    flag='MAKETHIN',
    class_='sextupole',
    slice_=1,
)

assert madx.command.select(
    flag='MAKETHIN',
    class_='sbend',
    slice_=1,
)

madx.command.makethin(
    makedipedge=False,
    style='simple',
    sequence='fodo',
)

fodo_sext = pysixtrack.Line.from_madx_sequence(madx.sequence.fodo)

In [ ]:
#for el in fodo_sext.elements:
#    print (el)

Going on with particle tracking:

In [ ]:
# define initial particle distribution & prepare recording array
particles = pysixtrack.Particles(x=x_ini, delta=delta)

rec_x = np.zeros((ncells, npart), dtype=float)
rec_x[0] = particles.x

In [ ]:
# tracking!
for i in range(1, ncells):
    fodo_sext.track(particles)
    rec_x[i] = particles.x

In [ ]:
plt.plot(rec_x[:, npart//2 + 1], lw=2, label='$\delta=0$')
plt.plot(rec_x[:, 0], lw=2, label='$\delta=-0.001$')

plt.xlabel('number of FODO cells')
plt.ylabel('$x$ [m]')
plt.legend(bbox_to_anchor=(1.05, 1));

In [ ]:
# evaluate tunes via NAFF
Qx_delta = np.zeros(npart, dtype=float)

for i in range(npart):
    Qx_delta[i] = PyNAFF.naff(rec_x[:, i], turns=ncells, nterms=1)[0, 1]

In [ ]:
plt.plot(1e3 * delta, Qx_delta)

plt.xlabel('$\delta$ [$10^{-3}$]')
plt.ylabel('$Q_x$');

The fit for the now much flatter slope of the tune change with $\delta$ gives:

In [ ]:
np.polyfit(delta, Qx_delta, 1)

$\implies$ $Q'_x=-0.01$ is nearly zero, i.e. the chromaticity correction scheme works! (The remainders are due to the thin-lens approximation!)

<i>Hint: we have used 2 sextupoles as 2 degrees of freedom to correct both the horizontal and the vertical chromaticity to zero. One could use only one sextupole degree of freedom, but then only one of the transverse planes can be corrected to $Q'=0$, the other one likely increases!</i>

<div style="text-align: center; width:100%;">
    <h2>Part II: Synchrotron Radiation</h2>
</div>

<h2>Synchrotron Radiation</h2>

<img src="./img/bending-synchrotronlight.webp" alt="Bending magnet synchrotron light principle" style="width:30%; float:right; margin: 1em;" />

Synchrotron light: electromagnetic radiation emitted by charged particles when accelerated in external magnetic field.

$\implies$ Could be from any source (e.g. astrophysical), but is generally thought of as coming from a man-made relativistic beam of electrons.

$\implies$ **Synchrotron radiation sources**:

- high-brightness photon beams
- tunability of photon spectrum
- pulsed time structure

<p style="clear: both; font-size: 10pt; text-align: right; float: right;">image by <a href="https://doi.org/10.1007/978-3-319-43866-5_3">T. Uruga</a></p>

<h2>Emitted power per radiating particle</h2>

Moving charged particle source $=$ time-varying charge and current density $\implies$ Maxwell equations and wave equation to find EM fields $\implies$ frequency distribution of radiation and critical frequency $\omega_c$ (median of spectrum)

<img src="./img/spectrum-wiggler.png" alt="Photon spectrum" style="width:25%; float: right; margin: 1em;" />

$$\omega_c=\cfrac{3}{2}\cfrac{c}{\rho}\gamma^3$$

$\implies$ Integrating over spectrum yields total radiated power <!--(at $\beta\approx 1$):-->

$$P_\gamma = \cfrac{e^2c}{6\pi\epsilon_0}\cfrac{\gamma^4}{\rho^2}$$

- decreasing bending radius $\rho$ (more bending!) increases critical frequency and power
- mass of particles enters via $\gamma=E_\text{tot}/m_0c^2$ $\implies$ $P_\gamma\propto 1/m_0^4$ <br />(electrons radiate ***much*** more than protons!)

<p style="clear: both; font-size: 10pt; text-align: right; float: right;">image by I. Martin / Diamond</p>

<h2>Two Types of Sources</h2>

Bending magnet (dipole) vs. insertion device (wiggler / undulator):

<div style="width: 80%; margin: 1em auto;">
<img src="./img/bending-synchrotronlight.webp" alt="Bending magnet synchrotron light principle" style="width: 30%; float: left; margin: 1em;" />
<img src="./img/insertion-device.webp" alt="Insertion device synchrotron light principle" style="width: 30%; float: left; margin: 1em;" />
    <p style="clear:both;"></p>
</div>

<p style="clear: both; font-size: 10pt; text-align: right; float: right;">images by <a href="https://doi.org/10.1007/978-3-319-43866-5_3">T. Uruga</a></p>

Advantage of insertion devices:

- alternating permanent magnets force electrons onto sinusoidal trajectories
- extremely focused, i.e. ultra-bright photon beams
- wigglers: stronger deflection, broad spectrum
- undulators: weaker deflection, waves interfere constructively, almost mono-chromatic **coherent** synchrotron light 

<h2>Synchrotron Radiation Facilities</h2>

<img src="./img/lightsource-schematic.webp" alt="Storage Ring based Light Source principle" style="width:30%; float:right; margin: 1em;" />

<p style="clear: both; font-size: 10pt; text-align: right; float: right;">image by <a href="https://doi.org/10.1007/978-3-319-43866-5_3">T. Uruga</a></p>

<img src="./img/linac-srsource.jpg" alt="Linac-based Light Source principle" style="width: 30%; float:right; clear:both; margin: 1em;" />

Two categories of synchrotron radiation facilities:

1. Storage rings
2. Linac-based sources

$\implies$ Free Electron Lasers: <br />photon beam intensity $I\propto N_e^2$ instead of $I\propto N_e$ for conventional sources

<h2>Future Circular lepton Collider FCC-ee</h2>

<img src="./img/fcc-footprint.png" alt="FCC-ee footprint" style="width:30%; float:right; margin: 1em auto;" />

<p style="clear: both; font-size: 10pt; text-align: right; float: right;">images by <a href="https://home.cern/science/accelerators/future-circular-collider/">CERN</a></p>

On the high-energy physics front: <br />next-generation facility for **high-precision** experiments

$\implies$ Higgs factory to study properties of Higgs <br />(discovered at LHC in 2012)

Collide leptons (electrons and positrons) for clean initial state during collision, much smaller background compared to proton colliders $\implies$ precision measurements!

*European Particle Physics Strategy Update* 2026: consider 91 km long FCC-ee proposal as top priority

$\implies$ Key issue of circular high-energy lepton colliders: **synchrotron radiation** (50 MW / beam!)

<h2>Synchrotron Radiation for Beam Size Detectors</h2>

<img src="./img/bsrt-meas.png" alt="Beam Size measurement with BSRT" style="width: 20%; float:right; margin: 1em;" />

LHC employs BSRT system: Beam Synchrotron Radiation Telescope

<img src="./img/bsrt.png" alt="BRST schematic" style="width: 60%; margin: 1em auto;" />

<p style="clear: both; font-size: 10pt; text-align: right; float: right;">images by <a href="https://indico.cern.ch/event/287930/contributions/660535/attachments/536481/739568/Wendt_LHC_SyncLightMonitor.pdf">F. Caspers</a></p>
    
$\implies$ At TeV level, protons do provide enough synchrotron radiation to be "seen"!

<h2>Summary</h2>

- Optical / Twiss functions and Dispersion of a FODO cell
- Dipoles generate & quadrupoles focuse dispersion function
- Quadrupole focal length depends on momentum, tune change, chromaticity
- Synchrotron radiation from accelerating/bending charged particles
- Light sources vs. HEP
<!--
- Chromatic tune change in a FODO cell
- Correction of chromaticity with sextupole magnets
-->

<h2>TT26 Survey</h2>

Thank you very much!

<img src="./img/survey-qrcode.png" alt="Survey QR code" style="width: 30%; margin: 1em auto;" />